# Verificação da PNS 2019 com os microdados reais (Google Colab)

Este *notebook* executa, sobre os microdados da PNS 2019 guardados no Google Drive, os dois *scripts* do repositório que dependem deles e compara o resultado com o que foi publicado no painel:

1. `reproducao/rodar_pns.py` — modelos preditivos (regressão logística, *Random Forest*, XGBoost) → compara com `pns_results.js`;
2. `reproducao/rodar_pns_raca.py` — prevalência de diabetes por cor ou raça → compara com `reproducao/fontes/pns/pns_raca.json` e calcula também a especificação do indicador oficial do IBGE.

**Como usar:** menu **Ambiente de execução → Executar tudo** e autorize o acesso ao Google Drive quando for pedido. Leva poucos minutos. O resumo é gravado na pasta `Dados` do Drive, em `verificacao_pns_colab.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os
PREFERIDO = '/content/drive/MyDrive/TCC /Dados/dados_brutos/PNS_2019.txt'
PNS = PREFERIDO if os.path.exists(PREFERIDO) else (glob.glob('/content/drive/MyDrive/**/dados_brutos/PNS_2019.txt', recursive=True) or [None])[0]
assert PNS, 'Não encontrei PNS_2019.txt dentro de uma pasta dados_brutos no Drive.'
PASTA_SAIDA = os.path.dirname(os.path.dirname(PNS))
print('Microdados:', PNS, f'({os.path.getsize(PNS)/1e6:.0f} MB)')

In [ ]:
RAMO = 'main'
!rm -rf /content/repo && git clone -q --depth 1 -b {RAMO} https://github.com/PiettroNaka/diabetes-tcc- /content/repo
os.makedirs('/content/repo/reproducao/dados_brutos', exist_ok=True)
os.symlink(PNS, '/content/repo/reproducao/dados_brutos/PNS_2019.txt')
!git -C /content/repo log --oneline -1

## 1. Modelos preditivos (`rodar_pns.py`)

In [ ]:
import json, subprocess, sklearn, xgboost, pandas, numpy
versoes = {'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__, 'pandas': pandas.__version__, 'numpy': numpy.__version__}
print(versoes)
!cd /content/repo && python reproducao/rodar_pns.py

carregar = lambda t: json.loads(t[t.index('{'):t.rindex('}') + 1])
novo = carregar(open('/content/repo/pns_results.js', encoding='utf-8').read())
publicado = carregar(subprocess.run(['git', '-C', '/content/repo', 'show', 'HEAD:pns_results.js'], capture_output=True, text=True).stdout)
comparacao_modelos = {k: novo[k] == publicado[k] for k in publicado}
for k, igual in comparacao_modelos.items():
    print(f'{k:26s}', 'IGUAL' if igual else f'DIFERENTE\n   novo:      {novo[k]}\n   publicado: {publicado[k]}')

## 2. Prevalência por cor ou raça (`rodar_pns_raca.py`)

In [ ]:
!cd /content/repo && python reproducao/rodar_pns_raca.py
raca = json.load(open('/content/repo/reproducao/pns_raca_reproduzida.json', encoding='utf-8'))

## 3. Resumo gravado no Drive

In [ ]:
resumo = {'versoes': versoes, 'commit': subprocess.run(['git', '-C', '/content/repo', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip(),
          'modelos_iguais_ao_publicado': comparacao_modelos, 'pns_results_reproduzido': novo, 'raca': raca}
destino = os.path.join(PASTA_SAIDA, 'verificacao_pns_colab.json')
json.dump(resumo, open(destino, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('Resumo gravado em:', destino)